# Python 클래스 파일별 3 · 8.ipynb 상속 · 오버라이딩 · 다형성 · `super()`

**한 줄 요약** · Dog · Cat · Bird로 상속 · 오버라이딩 · 다형성 · `super()`를 확인하고 `nn.Module` 모델로 연결

- **원본** · `2.ipynb/01_강의노트/8.ipynb` (원본 파일은 그대로 둠)
- **종합 정리본** · `00_최소선/02_Python클래스` 전체 정리 4장 · 5장

## 🎯 학습 목표

이 노트북을 끝내면 **자료를 보지 않고** 다음을 할 수 있어야 함

1. `class 자식(부모):`로 물려받고, 같은 이름의 메서드를 다시 정의(오버라이딩)한다
2. 같은 호출이 객체마다 다르게 동작하는 다형성을 예로 설명한다
3. `super()`로 부모 기능을 먼저 실행하고, 자식 `__init__` 첫 줄에 `super().__init__()`을 둔다
4. `nn.Module`을 물려받은 모델에서 `super().__init__()`을 빠뜨리면 생기는 에러를 말한다

## 어디에 쓰나

*실제 상황 → 이 권에서 배우는 것*

- **모델을 만들 때** · `class Net(nn.Module):` 상속 (3장)
- **`forward`를 직접 쓸 때** · 오버라이딩 (1-2 · 3장)
- **`nn.Sequential`이 층을 차례로 부를 때** · 다형성 — 같은 호출, 층마다 다른 계산 (1-2)
- **자식 `__init__`을 만들 때** · 첫 줄 `super().__init__()` (2-1 · 3장)

## 학습 지도

<img src="images/pfc3_map.png" alt="8.ipynb 학습 지도" width="763">

**예상 학습 시간** · 개념 50분 + 실습 20분

## 목차

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">장</th><th style="text-align: left;">제목</th><th style="text-align: left;">이 장의 질문</th></tr>
</thead>
<tbody>
<tr><td>0</td><td>준비</td><td>실행 환경 · 원본 구성</td></tr>
<tr><td>1</td><td>상속과 오버라이딩</td><td>부모 기능을 물려받아 그대로 쓰거나 일부만 바꾸려면?</td></tr>
<tr><td>2</td><td>super()로 부모 기능 이어 쓰기</td><td>부모 기능을 먼저 실행하고 덧붙이려면, 빠뜨리면 어떻게 되는가?</td></tr>
<tr><td>3</td><td>종합 실습</td><td>상속 · <code>super()</code> · 오버라이딩으로 PyTorch 모델을 만들려면?</td></tr>
<tr><td>—</td><td>마무리 · 부록</td><td>오늘 배운 것 · 기억 확인 · 오답 노트 · 면접 · 요약표 · 복습일 · 원본 대응표 · 전체 코드</td></tr>
</tbody></table>

---
## 0. 준비

1 · 2장은 기본 문법만, 3장은 PyTorch를 씀. **셀은 위에서부터 차례로 실행**(1-1의 `Animal`을 2장까지 씀).

*항목 → 설정*

- **라이브러리** · PyTorch (3장만)
- **seed · 장치** · 3장 `torch.manual_seed(0)` · CPU
- **데이터** · 원본의 동물 객체 · 3장은 `torch.ones` 입력

**실행 흐름**

1. `sys` · `torch` 불러오기
2. 파이썬 · PyTorch 버전 출력
3. 설치 확인

In [1]:
# 실행 환경 확인: 1 · 2장은 기본 문법, 3장만 PyTorch 사용
# 왜 · 3장에서 쓰는 PyTorch가 설치됐는지 미리 확인해 중간에 멈추지 않게 하기 위해
import sys  # 파이썬 버전 확인용

import torch  # 3장 nn.Module 실습

print("Python:", sys.version.split()[0])            # 3.10 이상이면 됨
print("PyTorch:", torch.__version__.split("+")[0])  # 설치 확인

Python: 3.14.6
PyTorch: 2.11.0


### 원본 구성

- 원본 학습 목표 4개
  - 자식 클래스가 부모 메서드를 사용
  - 자식 클래스에서 부모 메서드를 다시 정의 (오버라이딩)
  - 같은 메서드가 객체에 따라 다르게 동작 (다형성)
  - `super()`로 부모 메서드 호출

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">원본 절</th><th style="text-align: left;">내용</th><th style="text-align: left;">이 노트북</th></tr>
</thead>
<tbody>
<tr><td>1 ~ 2</td><td>부모 클래스 · 그대로 상속</td><td>1-1</td></tr>
<tr><td>3 ~ 4</td><td>오버라이딩 · 다형성</td><td>1-2</td></tr>
<tr><td>5 ~ 6</td><td><code>super()</code>로 메서드 확장 · 생성자에서 <code>super()</code></td><td>2-1</td></tr>
<tr><td>7</td><td>객체와 클래스의 관계 (<code>isinstance</code>)</td><td>2-2</td></tr>
<tr><td>보충</td><td><code>super()</code>를 빠뜨리면 · <code>nn.Module</code>에서 확인</td><td>2-2 · 3</td></tr>
</tbody></table>

- 원본 끝의 `dir()` 출력 셀(속성 목록 70여 개)은 제외 → 메서드를 찾는 순서(`__mro__`)로 대신 확인

**원본 힌트**

- 원본은 Animal 하나를 Dog · Cat · Bird가 물려받는 흐름
- 그대로 쓰기 → 바꿔 쓰기(오버라이딩) → 부모 것 + 덧붙이기(`super()`) 순서
- `nn.Module`을 이해하려면 이 파일과 파일별 2(2.ipynb)가 특히 중요

---
## 1. 상속과 오버라이딩

> **이 장의 흐름**
>
> - 원본 1 ~ 2절: 부모 Animal과 아무것도 안 바꾼 자식
> - 원본 3 ~ 4절: sound()를 다시 정의 → 같은 호출, 다른 결과

<img src="images/cls2_inherit.png" alt="상속 관계와 메서드를 찾는 순서" width="763">

**파이프라인 위치** · ④ 모델 — `nn.Module` 물려받기

### 1-1. `A` 부모 클래스와 그대로 상속 · `class 자식(부모):` · `pass` <sub>원본 1 ~ 2절</sub>

> 🎯 **아무 메서드도 없는 자식 클래스는 무엇을 쓸 수 있는가?**

💡 **쉽게 말하면**

- 상속은 **부모 레시피를 물려받아 쓰는 것**.
- 자식에 없으면 파이썬이 **부모에서 찾아** 씀.

**실행 흐름**

1. Animal 정의
2. BasicDog(pass)
3. 두 객체의 메서드 호출

**먼저 예측** · 아무 메서드도 없는 `BasicDog`의 `sound()`는? — 답은 코드 실행 결과 아래에

In [2]:
# 왜 · 여러 동물이 함께 쓰는 속성 · 기능을 부모 한 곳에 모아 중복을 없애기 위해
class Animal:                  # 부모 클래스
    def __init__(self, name):  # 모든 동물의 이름
        self.name = name       # 자식도 물려받는 속성

    def introduce(self):                        # 자식이 그대로 쓸 메서드
        return f"제 이름은 {self.name}입니다."  # 이름으로 소개

    def sound(self):                    # 자식이 바꿀 메서드
        return "동물이 소리를 냅니다."  # 기본 소리


# 왜 · 몸체가 비어도 부모 기능을 모두 물려받는다는 것을 확인하기 위해
class BasicDog(Animal):  # 2절: 아무것도 쓰지 않고 물려받기
    pass                 # 몸체 없음


# 왜 · 부모 객체와 자식 객체가 같은 메서드를 쓸 수 있는지 비교하기 위해
animal = Animal("동물")                          # 부모 객체
basic_dog = BasicDog("초코")                     # 자식 객체
print(animal.introduce(), animal.sound())        # 부모 메서드
print(basic_dog.introduce(), basic_dog.sound())  # 자식에 없으면 부모에서 찾음

제 이름은 동물입니다. 동물이 소리를 냅니다.
제 이름은 초코입니다. 동물이 소리를 냅니다.


**결과** · 예측 정답은 `동물이 소리를 냅니다.` — 부모 것을 그대로 씀

- `제 이름은 동물입니다. 동물이 소리를 냅니다.` — 부모 객체
- `제 이름은 초코입니다. 동물이 소리를 냅니다.` — BasicDog에 없으니 Animal에서 찾음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>class 자식(부모):</code></td><td>부모의 속성 · 메서드를 물려받음</td><td>공통 기능이 있는 여러 클래스</td></tr>
<tr><td><code>pass</code>로 그대로 상속</td><td>부모 메서드를 <strong>그대로</strong> 사용</td><td>이름만 다르고 동작은 같을 때</td></tr>
<tr><td>상속 없이 복사</td><td>같은 코드를 클래스마다 반복</td><td>쓰면 안 됨 (고칠 곳이 늘어남)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - `__init__`도 물려받으므로 `BasicDog("초코")`처럼 부모 `__init__`의 인자를 그대로 줘야 함

> ✅ **한 줄 정리** · `class 자식(부모):`로 물려받으면 자식에 없는 메서드는 부모에서 찾아 쓴다.

**딥러닝 코드에서는**

- `class Net(nn.Module):` — `parameters()` · `to()` · `train()`을 물려받음

### 1-2. `A` 오버라이딩과 다형성 · `def sound(self)` · `for a in animals` <sub>원본 3 ~ 4절</sub>

> 🎯 **같은 `sound()` 호출이 객체마다 다른 결과를 내는 이유는?**

💡 **쉽게 말하면**

- 오버라이딩은 **물려받은 레시피의 한 단계를 바꿔 쓰기**.
- 다형성은 **같은 주문, 가게마다 다른 요리**.

**실행 흐름**

1. Dog · Cat이 sound() 다시 정의
2. dog의 두 메서드
3. 종류가 다른 객체 목록을 같은 호출로

**먼저 예측** · `[Animal, Dog, Cat]` 객체에 똑같이 `sound()`를 부르면? — 답은 코드 실행 결과 아래에

In [3]:
# 왜 · 부모 메서드를 같은 이름으로 다시 정의해 자식만의 동작을 주기 위해
class Dog(Animal):     # Animal을 물려받음
    def sound(self):   # 같은 이름으로 다시 정의 (오버라이딩)
        return "멍멍"  # Dog만의 소리


# 왜 · 같은 부모를 둔 다른 자식이 각자 다르게 동작하는 것을 보이기 위해
class Cat(Animal):     # Animal을 물려받음
    def sound(self):   # 오버라이딩
        return "야옹"  # Cat만의 소리


dog, cat = Dog("콩이"), Cat("나비")  # 자식 객체 둘
print(dog.introduce(), dog.sound())  # introduce는 부모 것, sound는 Dog 것


# 4절 다형성: 같은 호출 sound() → 객체마다 다른 결과
# 왜 · 종류를 따지는 if 없이 같은 코드로 여러 객체를 처리하기 위해
animals = [Animal("동물"), Dog("초코"), Cat("나비")]         # 종류가 다른 객체
for current_animal in animals:                               # 종류를 묻지 않고 반복
    print(current_animal.name, "→", current_animal.sound())  # 같은 호출

제 이름은 콩이입니다. 멍멍
동물 → 동물이 소리를 냅니다.
초코 → 멍멍
나비 → 야옹


**결과** · 예측 정답은 객체마다 다른 소리 — 다형성

- `제 이름은 콩이입니다. 멍멍` — introduce는 물려받고 sound는 Dog 것
- 같은 `sound()` 호출, 객체마다 다른 결과
  - `동물 → 동물이 소리를 냅니다.`
  - `초코 → 멍멍`
  - `나비 → 야옹`

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>오버라이딩</td><td>같은 이름으로 <strong>완전히 교체</strong></td><td>자식마다 동작이 달라야 할 때 (<code>forward</code>)</td></tr>
<tr><td>다형성</td><td>같은 호출 → 객체마다 다른 동작</td><td>종류를 묻지 않고 반복할 때</td></tr>
<tr><td><code>if type(x) is Dog:</code> 분기</td><td>종류마다 if로 나눔</td><td>쓰지 않는 것이 좋음 (다형성으로 대체)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 메서드 이름 철자가 부모와 다르면(`sounds`) 오버라이딩이 아니라 새 메서드가 됨 → 부모 것이 그대로 불림

> ✅ **한 줄 정리** · 같은 이름의 메서드를 자식에서 다시 정의하면 같은 호출이 객체마다 다르게 동작한다.

**딥러닝 코드에서는**

- `for layer in layers: x = layer(x)` — 층 종류를 묻지 않고 같은 호출 (`nn.Sequential`)

---
## 2. super()로 부모 기능 이어 쓰기

> **이 장의 흐름**
>
> - 원본 5 ~ 6절: `super()`로 메서드 확장 · 생성자에서 `super().__init__()`
> - 원본 7절 + 보충: `isinstance` · 찾는 순서 · `super()`를 빠뜨린 자식

<img src="images/cls2_super.png" alt="super()로 부모 초기화 후 자식 속성 추가" width="763">

**파이프라인 위치** · ④ 모델 — `super().__init__()`

### 2-1. `A` 부모 기능 확장 · `super().메서드()` · `super().__init__()` <sub>원본 5 ~ 6절</sub>

> 🎯 **부모 메서드 결과에 덧붙이고, 부모 초기화 뒤 자식 속성을 추가하려면?**

💡 **쉽게 말하면**

- `super()`는 **부모 방식대로 먼저 하고 덧붙이기**.
- 자식 `__init__`이 있으면 부모 `__init__`은 자동으로 안 돎 → 직접 부름.

**실행 흐름**

1. LoudDog: 부모 sound + !!!
2. Bird: `super().__init__` + can_fly
3. Bird 메서드 · `__dict__`

**먼저 예측** · `LoudDog("두부").sound()`는? — 답은 코드 실행 결과 아래에

In [4]:
# 왜 · 부모 동작을 버리지 않고 super()로 받아 그 위에 덧붙이기 위해
class LoudDog(Dog):                     # Dog를 물려받음
    def sound(self):                    # 부모 sound를 확장
        parent_sound = super().sound()  # 부모(Dog)의 sound() → 멍멍
        return parent_sound + "!!!"     # 덧붙이기


# 왜 · 자식에게 필요한 값만 추가하고 공통 초기화는 부모에게 맡기기 위해
class Bird(Animal):                     # Animal을 물려받음
    def __init__(self, name, can_fly):  # 자식은 인자가 하나 더
        super().__init__(name)          # name 저장은 부모에게 맡김
        self.can_fly = can_fly          # Bird에만 필요한 속성

    def sound(self):   # 오버라이딩
        return "짹짹"  # Bird만의 소리

    # 왜 · 자식에만 있는 속성을 쓰는 기능은 자식 클래스에 두기 위해
    def show_flight(self):                          # 자식에만 있는 메서드
        if self.can_fly:                            # 날 수 있으면
            return f"{self.name}: 날 수 있습니다."  # 원본 문장
        return f"{self.name}: 날 수 없습니다."      # 아니면


# 왜 · super()로 확장한 결과와 추가 속성이 제대로 동작하는지 확인하기 위해
print(LoudDog("두부").sound())                             # 부모 결과 + !!!
bird = Bird("참새", True)                                  # name은 부모, can_fly는 자식이 저장
print(bird.introduce(), bird.sound(), bird.show_flight())  # 물려받은 · 바꾼 · 새 메서드
print(bird.__dict__)                                       # 두 속성 모두 저장됨

멍멍!!!
제 이름은 참새입니다. 짹짹 참새: 날 수 있습니다.
{'name': '참새', 'can_fly': True}


**결과** · 예측 정답은 `멍멍!!!`

- `멍멍!!!` — `super().sound()`가 Dog의 멍멍을 가져와 덧붙임
- `제 이름은 참새입니다. 짹짹 참새: 날 수 있습니다.` — 물려받은 · 바꾼 · 새 메서드
- `{'name': '참새', 'can_fly': True}` — name은 부모 `__init__`이, can_fly는 자식이 저장

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>super().메서드()</code></td><td>부모 결과를 받아 <strong>덧붙임</strong></td><td>부모 기능 + 추가 (<code>LoudDog</code>)</td></tr>
<tr><td><code>super().__init__(...)</code></td><td>부모 속성 준비 <strong>후</strong> 자식 속성 추가</td><td>자식에 <code>__init__</code>이 있으면 <strong>항상</strong></td></tr>
<tr><td><code>Animal.__init__(self, name)</code></td><td>부모 이름을 직접 적어 호출</td><td>옛 방식 · 부모가 바뀌면 고칠 곳이 늘어남</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - `super().__init__()`은 자식 `__init__`의 **첫 줄**에 → 부모가 준비한 속성을 자식이 바로 쓸 수 있음

> ✅ **한 줄 정리** · `super()`로 부모 기능을 먼저 실행한 뒤 자식만의 기능 · 속성을 덧붙인다.

**딥러닝 코드에서는**

- `super().__init__()` → `self.conv = ...` — 모든 PyTorch 모델의 첫 두 줄

### 2-2. `B` 객체와 클래스의 관계 · `super()`를 빠뜨리면 · `isinstance()` · `__mro__` <sub>원본 7절 + 보충</sub>

> 🎯 **자식 객체는 부모의 객체이기도 한가, 부모 초기화를 빠뜨리면 어떻게 되는가?**

💡 **쉽게 말하면**

- Dog 객체는 **Animal이기도 함** (강아지도 동물).
- 부모 초기화를 건너뛰면 부모가 만들 속성(name)이 **없음**.

**실행 흐름**

1. isinstance 3가지
2. 메서드를 찾는 순서
3. `super().__init__()` 생략한 자식

**먼저 예측** · `isinstance(dog, Animal)`은? — 답은 코드 실행 결과 아래에

In [5]:
# 왜 · 자식 객체는 부모 종류로도 인정된다는 것과 메서드 찾는 순서를 확인하기 위해
print(isinstance(dog, Dog), isinstance(dog, Animal), isinstance(cat, Dog))  # 자식 = 부모 객체
print([c.__name__ for c in LoudDog.__mro__])                                # 메서드를 찾는 순서


# 왜 · super().__init__()을 빠뜨리면 부모 속성이 생기지 않는 버그를 재현하기 위해
class BrokenBird(Animal):               # super().__init__()을 빠뜨린 자식
    def __init__(self, name, can_fly):  # 부모 초기화 생략
        self.can_fly = can_fly          # name이 저장되지 않음


try:
    BrokenBird("펭귄", False).introduce()  # name을 쓰는 순간
except AttributeError as e:                # name 속성이 없음
    print("AttributeError:", e)            # 에러 문장

True True False
['LoudDog', 'Dog', 'Animal', 'object']
AttributeError: 'BrokenBird' object has no attribute 'name'


**결과** · 예측 정답은 `True` — Dog 객체는 Animal의 객체이기도 함

- `True True False` — dog는 Dog이자 Animal, cat은 Dog가 아님
- `['LoudDog', 'Dog', 'Animal', 'object']` — 자식 → 부모 → object 순서로 찾음
- `AttributeError: 'BrokenBird' object has no attribute 'name'` — 부모 초기화를 건너뛰어 name이 없음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>isinstance(x, 부모)</code></td><td>상속까지 인정 → True</td><td>자료형 확인 (기본 선택)</td></tr>
<tr><td><code>type(x) is 클래스</code></td><td>정확히 그 클래스만</td><td>상속을 제외해야 할 때</td></tr>
<tr><td><code>클래스.__mro__</code></td><td>메서드를 찾는 순서</td><td>어느 클래스 메서드가 불리는지 확인</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - BrokenBird는 만들 때는 멀쩡하고 `name`을 쓰는 순간에야 에러 → 원인과 에러 위치가 떨어져 있음

> ✅ **한 줄 정리** · 자식 객체는 부모의 객체이기도 하고, `super().__init__()`을 빠뜨리면 부모 속성이 없다.

**딥러닝 코드에서는**

- `isinstance(layer, nn.Linear)` — 층 종류별로 가중치 초기화할 때

---
## 3. 종합 실습 · nn.Module 모델 만들기

> **이 장의 흐름**
>
> - 1 · 2장 문법을 한 번에 사용
> - `nn.Module`을 물려받은 Net과 `super()`를 빠뜨린 BrokenNet 비교

<img src="images/pfc3_net.png" alt="Net은 nn.Module을 물려받아 forward만 다시 정의" width="763">

**파이프라인 위치** · ④ 모델

> 🎯 **`nn.Module`을 물려받아 층을 등록하고 `forward`를 다시 정의한 모델을 만들려면?**

💡 **쉽게 말하면**

- **상속**(nn.Module) → **super**(부모 준비) → **self.층** → **forward 오버라이딩**.
- 만들지 않은 `parameters()`도 물려받아 씀.

**실행 흐름**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">단계</th><th style="text-align: left;">하는 일</th><th style="text-align: left;">쓰는 문법</th></tr>
</thead>
<tbody>
<tr><td>1</td><td>Net(nn.Module)</td><td>1-1 상속</td></tr>
<tr><td>2</td><td><code>super().__init__()</code> · 층 등록</td><td>2-1 생성자에서 super</td></tr>
<tr><td>3</td><td>forward 다시 정의 · <code>model(x)</code></td><td>1-2 오버라이딩</td></tr>
<tr><td>4</td><td>super 생략한 BrokenNet</td><td>2-2 빠뜨리면</td></tr>
</tbody></table>

**먼저 예측** · `super().__init__()` 없이 `self.fc = nn.Linear(...)`를 쓰면 언제 에러가 날까? — 답은 코드 실행 결과 아래에

In [6]:
import torch          # 텐서
from torch import nn  # 층 · 모델


# 1 ~ 3단계: 상속 → super → 층 등록 → forward 오버라이딩
# 왜 · PyTorch 모델이 바로 이 상속 · super · 오버라이딩 규칙으로 만들어진다는 것을 보이기 위해
class Net(nn.Module):              # nn.Module을 물려받음
    def __init__(self):            # 층 준비
        super().__init__()         # 부모(nn.Module) 준비를 먼저
        self.fc = nn.Linear(4, 2)  # 층 등록

    def forward(self, x):  # 오버라이딩: 계산 방법만 정의
        return self.fc(x)  # 입력 4 → 점수 2


# 왜 · 가중치 초깃값을 고정해 같은 PC · 같은 버전에서 출력이 같게 하기 위해
torch.manual_seed(0)                                                       # 가중치 초기값 고정
net = Net()                                                                # 모델 객체
# 상속 확인 · model(x) 출력 모양
print(isinstance(net, nn.Module), tuple(net(torch.ones(3, 4)).shape))
print("물려받은 parameters():", sum(p.numel() for p in net.parameters()))  # 4 × 2 + 2


# 4단계: super().__init__()을 빠뜨린 모델
# 왜 · 모델에서 가장 흔한 실수가 어떤 에러로 나타나는지 미리 보기 위해
class BrokenNet(nn.Module):        # super().__init__() 생략
    def __init__(self):            # 부모 준비 없이
        self.fc = nn.Linear(4, 2)  # 층 저장 → 즉시 에러


try:
    BrokenNet()                  # 만드는 순간
except AttributeError as e:      # 모듈을 등록할 공간이 없음
    print("AttributeError:", e)  # 에러 문장

True (3, 2)
물려받은 parameters(): 10
AttributeError: cannot assign module before Module.__init__() call


**결과** · 예측 정답은 만드는 순간 — 층을 저장하는 줄에서 바로 에러 (2-2 BrokenBird는 name을 쓸 때)

- `True (3, 2)` — Net은 nn.Module의 객체 · 배치 3 × 점수 2
- `물려받은 parameters(): 10` — 만들지 않은 메서드도 물려받음 · 4 × 2 + 2 = 10
- `AttributeError: cannot assign module before Module.__init__() call` — nn.Module은 super 생략을 즉시 알려 줌

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>class Net(nn.Module)</code></td><td>상속</td><td><code>parameters()</code> · <code>to()</code> · <code>train()</code> 물려받기</td></tr>
<tr><td><code>super().__init__()</code></td><td>부모 준비</td><td>층 등록 공간 만들기</td></tr>
<tr><td><code>def forward(self, x)</code></td><td>오버라이딩</td><td>계산 방법만 직접 정의</td></tr>
<tr><td><code>net(x)</code></td><td>부모의 호출 → 내 forward</td><td><code>net.forward(x)</code> 대신 항상 <code>net(x)</code></td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - `net.forward(x)`로 직접 부르면 nn.Module의 hook 같은 부가 기능을 건너뜀 → 항상 `net(x)`

> ✅ **한 줄 정리** · 모델은 `nn.Module` 상속 → `super().__init__()` → `self.층` 등록 → `forward` 재정의 순서로 만든다.

**딥러닝 코드에서는**

- CNN · RNN · Transformer 모두 같은 틀 — 바뀌는 것은 `__init__`의 층과 `forward`의 계산 순서뿐

---
## 그래서 오늘 무엇을 배웠는가?

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">학습 목표</th><th style="text-align: left;">할 수 있게 된 것</th><th style="text-align: left;">확인 위치</th></tr>
</thead>
<tbody>
<tr><td>1. 그대로 상속</td><td>BasicDog에 없으면 Animal에서 찾음</td><td>1-1</td></tr>
<tr><td>2. 오버라이딩 · 다형성</td><td>같은 <code>sound()</code> → 멍멍 · 야옹</td><td>1-2</td></tr>
<tr><td>3. super()</td><td>멍멍!!! · Bird의 name은 부모가 저장</td><td>2-1</td></tr>
<tr><td>4. 관계 · 생략</td><td><code>isinstance(dog, Animal)</code> True · BrokenBird AttributeError</td><td>2-2</td></tr>
<tr><td>5. nn.Module</td><td>parameters 10개 · BrokenNet은 만드는 순간 에러</td><td>3</td></tr>
</tbody></table>

### 주제별 한 줄 · 반드시 기억할 것

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">등급</th><th style="text-align: left;">주제</th><th style="text-align: left;">반드시 기억할 것</th><th style="text-align: left;">딥러닝 코드에서</th></tr>
</thead>
<tbody>
<tr><td><code>A</code></td><td>상속 <code>class 자식(부모)</code></td><td>부모 기능 물려받기</td><td><code>class Net(nn.Module)</code></td></tr>
<tr><td><code>A</code></td><td>오버라이딩</td><td>같은 이름으로 다시 정의</td><td><code>def forward(self, x)</code></td></tr>
<tr><td><code>A</code></td><td><code>super().__init__()</code></td><td>부모 준비 먼저</td><td>모든 모델의 첫 줄</td></tr>
<tr><td><code>B</code></td><td>다형성 · <code>isinstance</code></td><td>같은 호출 · 상속까지 인정</td><td><code>nn.Sequential</code> · 층별 초기화</td></tr>
</tbody></table>

---
## 기억 확인 문제

위 내용을 **보지 않고** 답한 뒤, ✅ 한 줄 정리와 비교해 ○ (정확) · △ (일부) · × (모름)를 표시. △ · ×인 섹션만 다시 보기.

### 1. 질문에 답하기

*섹션 → 🎯 질문*

- **1-1** · 자식에 없는 메서드는 어디서 찾나?
- **1-2** · 다형성이란?
- **2-1** · 자식 `__init__` 첫 줄에 둘 것은?
- **2-2** · `isinstance(dog, Animal)`은?
- **3** · `nn.Module`에서 `super().__init__()`을 빠뜨리면?

#### 내 답 · 이 셀을 더블클릭해서 작성

**1-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

### 2. 코드로 답하기

아래 빈 셀에 직접 작성 후 실행.

1. `Animal`을 물려받아 `sound()`가 `"음메"`인 `Cow`를 만들고 `introduce()` · `sound()` 출력
2. `LoudCat(Cat)`을 만들어 `super()`로 `"야옹?"`을 돌려주게 하기

In [7]:
# 코드 문제 직접 작성

### 3. 변형 문제 · 조건을 바꿔 다시 풀기

1. `Bird`를 물려받아 `can_fly=False`가 기본인 `Penguin` 만들기
2. `isinstance(LoudDog("a"), Dog)`의 결과는?
3. `Net`에 `nn.ReLU()`를 추가해 `forward`에서 `fc` 뒤에 적용하기

In [8]:
# 변형 문제 직접 작성

### 4. 모범 답안 · 1 ~ 3을 다 푼 뒤에 보기

#### 질문 답

*섹션 → 답*

- **1-1** · 부모 클래스 (`__mro__` 순서)
- **1-2** · 같은 호출이 객체마다 다르게 동작
- **2-1** · `super().__init__(...)`
- **2-2** · True — 자식 객체는 부모의 객체
- **3** · 층 저장 순간 AttributeError

#### 코드 문제 답

```python
class Cow(Animal):
    def sound(self):
        return "음메"


class LoudCat(Cat):
    def sound(self):
        return super().sound() + "?"


print(Cow("누렁이").introduce(), Cow("누렁이").sound())  # 제 이름은 누렁이입니다. 음메
print(LoudCat("나비").sound())  # 야옹?
```

#### 변형 문제 답

*문제 → 답*

- **1** · `def __init__(self, name): super().__init__(name, False)`
- **2** · `True` — LoudDog은 Dog의 자식
- **3** · `__init__`에 `self.act = nn.ReLU()` · `forward`에서 `return self.act(self.fc(x))`

---
## 오답 노트

틀린 문제는 정답을 베끼지 말고 **원인**을 적음. 다음 복습 때 이 표의 "재발 방지 질문"부터 스스로에게 물어보기.

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">틀린 내용</th><th style="text-align: left;">틀린 이유</th><th style="text-align: left;">올바른 원칙</th><th style="text-align: left;">재발 방지 질문</th></tr>
</thead>
<tbody>
<tr><td>(예) 자식 <code>__init__</code>에서 <code>super().__init__()</code>을 빠뜨림</td><td>자식에 <code>__init__</code>을 만들어도 부모 것이 자동 실행된다고 착각</td><td>자식 <code>__init__</code>이 부모 것을 가림 → 직접 호출</td><td>자식 <code>__init__</code> 첫 줄에 <code>super().__init__()</code>이 있나?</td></tr>
<tr><td></td><td></td><td></td><td></td></tr>
<tr><td></td><td></td><td></td><td></td></tr>
</tbody></table>

---
## 면접 30초 답변

**Q1. 상속과 오버라이딩은 무엇인가요?**
> - 상속은 부모 클래스의 속성과 메서드를 자식이 물려받는 것입니다.
> - 오버라이딩은 물려받은 메서드를 자식에서 같은 이름으로 다시 정의해 동작을 바꾸는 것입니다.
> - PyTorch 모델은 `nn.Module`을 상속하고 `forward`를 오버라이딩합니다.

**Q2. `super().__init__()`은 왜 필요한가요?**
> - 자식에 `__init__`을 만들면 부모 `__init__`이 자동으로 실행되지 않습니다.
> - 그래서 부모가 준비할 속성이 빠지는데, `nn.Module`은 층을 등록할 공간이 없어 층을 저장하는 순간 AttributeError가 납니다.

**Q3. 다형성의 예를 들어 주세요.**
> - Dog와 Cat에 똑같이 `sound()`를 부르면 각자 다른 소리를 냅니다.
> - `nn.Sequential`도 층 종류를 묻지 않고 차례로 `layer(x)`를 부르는데 층마다 다른 계산을 합니다.

---
## 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>class 자식(부모):</code> · <code>pass</code></td><td>Python 기본 문법</td><td>상속</td><td>1-1 <code>class BasicDog(Animal):</code></td></tr>
<tr><td>오버라이딩</td><td>Python 기본 문법</td><td>같은 이름으로 다시 정의</td><td>1-2 <code>def sound(self): return "멍멍"</code></td></tr>
<tr><td><code>super()</code></td><td>Python 내장 함수</td><td>부모 메서드 · <code>__init__</code> 호출</td><td>2-1 <code>super().__init__(name)</code></td></tr>
<tr><td><code>isinstance()</code> · <code>__mro__</code> · <code>AttributeError</code></td><td>Python 내장 함수 · 속성 · 예외</td><td>관계 확인 · 찾는 순서 · 속성 없음</td><td>2-2 <code>LoudDog.__mro__</code></td></tr>
<tr><td><code>nn.Module</code> · <code>nn.Linear</code> · <code>parameters()</code></td><td>torch.nn</td><td>모델 부모 · 층 · 물려받은 메서드</td><td>3 <code>class Net(nn.Module):</code></td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>torch.manual_seed</code> · <code>numel()</code></td><td>torch · Tensor</td><td>초기값 고정 · 원소 수</td><td>3 <code>sum(p.numel() for p in net.parameters())</code></td></tr>
</tbody></table>

### 할 것 · 하지 말 것

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">✅ 할 것</th><th style="text-align: left;">⚠️ 하지 말 것</th></tr>
</thead>
<tbody>
<tr><td>자식 <code>__init__</code> 첫 줄에 <code>super().__init__()</code></td><td>부모 초기화 생략</td></tr>
<tr><td>자료형 확인은 <code>isinstance</code></td><td><code>if type(x) is Dog:</code> 분기 남발</td></tr>
<tr><td>모델 호출은 <code>net(x)</code></td><td><code>net.forward(x)</code> 직접 호출</td></tr>
</tbody></table>

### 헷갈리는 짝 한눈에 보기

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">짝</th><th style="text-align: left;">차이</th><th style="text-align: left;">결론</th></tr>
</thead>
<tbody>
<tr><td>상속 vs 오버라이딩</td><td>물려받기 vs 다시 정의</td><td>필요한 것만 다시 정의</td></tr>
<tr><td>오버라이딩 vs <code>super()</code> 확장</td><td>교체 vs 부모 + 덧붙임</td><td>부모 기능이 필요하면 super</td></tr>
<tr><td><code>isinstance</code> vs <code>type() is</code></td><td>상속 인정 vs 정확히 그 클래스</td><td>대부분 isinstance</td></tr>
<tr><td>BrokenBird vs BrokenNet</td><td>쓸 때 에러 vs 만들 때 에러</td><td>둘 다 super 누락</td></tr>
</tbody></table>

---
## 다음 복습일

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">언제</th><th style="text-align: left;">할 일</th><th style="text-align: left;">걸리는 시간</th></tr>
</thead>
<tbody>
<tr><td>내일</td><td>기억 확인 문제 1을 자료 없이 다시 답하고 오답 노트 갱신</td><td>10분</td></tr>
<tr><td>3일 뒤</td><td>변형 문제 3개 다시 풀기 · Animal · Dog · Bird를 직접 다시 쓰기</td><td>15분</td></tr>
<tr><td>7일 뒤</td><td>3장 Net에 층 하나를 더 추가해 parameters 수를 손으로 계산 후 확인</td><td>20분</td></tr>
<tr><td>14일 뒤</td><td>오답 노트에 남은 문제만 · 전체 정리 5장(nn.Module 종합 실습) 전 ✅ 줄만 읽기</td><td>10분</td></tr>
</tbody></table>

---
## 부록

### 이 노트북을 만든 방법

- 1장 · 원본 강의노트(`8.ipynb`) 1 ~ 4절
- 2장 · 5 ~ 7절 + `super()` 누락 보충, 3장 · `nn.Module` 실습을 새로 추가
- 모든 코드는 실행한 출력과 함께 저장 · 형식 v2.1 (FORMAT_V2.md 1 ~ 7절)

### 읽는 법

*표시 → 의미*

- **`A` · `B`** · **A** 딥러닝 코드에 바로 나오는 필수 · **B** 알아두면 되는 것
- **🎯** · 이번 섹션에서 답해야 할 질문
- **💡** · 쉬운 설명 · 비유
- **⚠️** · 자주 하는 실수
- **✅** · 기억할 결론 = 🎯 질문의 답

- **섹션 앞부분** · 🎯 질문 → 💡 쉬운 설명 → 실행 흐름 → 먼저 예측 → 코드
- **섹션 뒷부분** · 결과(예측 정답 · 출력 줄) → 차이 · 언제 쓰나 → ⚠️ 주의 → ✅ 한 줄 정리
- **코드 주석** · 코드 위는 이 단계에서 하는 일, 코드 옆은 그 줄이 필요한 이유 (줄마다 · 열 맞춤)

### 원본 → 섹션 대응표

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">원본</th><th style="text-align: left;">내용</th><th style="text-align: left;">이 노트북</th></tr>
</thead>
<tbody>
<tr><td>1 ~ 2절 부모 · 그대로 상속</td><td>부모 클래스와 그대로 상속</td><td>1-1</td></tr>
<tr><td>3 ~ 4절 오버라이딩 · 다형성</td><td>오버라이딩과 다형성</td><td>1-2</td></tr>
<tr><td>5 ~ 6절 super()</td><td>super()로 확장 · 생성자에서 super()</td><td>2-1</td></tr>
<tr><td>7절 isinstance + 보충</td><td>관계 · super()를 빠뜨리면</td><td>2-2</td></tr>
<tr><td>보충 nn.Module</td><td>nn.Module 모델 만들기</td><td>3</td></tr>
</tbody></table>

### 전체 코드

1 ~ 3장 코드를 장별 셀로 모음. **위에서부터 이 셀들만 실행해도 전체가 동작**하며, 각 블록 머리 주석의 번호가 섹션.

**실행 흐름**

1. 1-1 부모 `Animal` · 빈 자식 `BasicDog`
2. 1-2 오버라이딩 · 다형성
3. 블록마다 결과 출력

In [9]:
# ---------- 1-1 ----------
# 왜 · 여러 동물이 함께 쓰는 속성 · 기능을 부모 한 곳에 모아 중복을 없애기 위해
class Animal:                  # 부모 클래스
    def __init__(self, name):  # 모든 동물의 이름
        self.name = name       # 자식도 물려받는 속성

    def introduce(self):                        # 자식이 그대로 쓸 메서드
        return f"제 이름은 {self.name}입니다."  # 이름으로 소개

    def sound(self):                    # 자식이 바꿀 메서드
        return "동물이 소리를 냅니다."  # 기본 소리


# 왜 · 몸체가 비어도 부모 기능을 모두 물려받는다는 것을 확인하기 위해
class BasicDog(Animal):  # 2절: 아무것도 쓰지 않고 물려받기
    pass                 # 몸체 없음


# 왜 · 부모 객체와 자식 객체가 같은 메서드를 쓸 수 있는지 비교하기 위해
animal = Animal("동물")                          # 부모 객체
basic_dog = BasicDog("초코")                     # 자식 객체
print(animal.introduce(), animal.sound())        # 부모 메서드
print(basic_dog.introduce(), basic_dog.sound())  # 자식에 없으면 부모에서 찾음


# ---------- 1-2 ----------
# 왜 · 부모 메서드를 같은 이름으로 다시 정의해 자식만의 동작을 주기 위해
class Dog(Animal):     # Animal을 물려받음
    def sound(self):   # 같은 이름으로 다시 정의 (오버라이딩)
        return "멍멍"  # Dog만의 소리


# 왜 · 같은 부모를 둔 다른 자식이 각자 다르게 동작하는 것을 보이기 위해
class Cat(Animal):     # Animal을 물려받음
    def sound(self):   # 오버라이딩
        return "야옹"  # Cat만의 소리


dog, cat = Dog("콩이"), Cat("나비")  # 자식 객체 둘
print(dog.introduce(), dog.sound())  # introduce는 부모 것, sound는 Dog 것


# 4절 다형성: 같은 호출 sound() → 객체마다 다른 결과
# 왜 · 종류를 따지는 if 없이 같은 코드로 여러 객체를 처리하기 위해
animals = [Animal("동물"), Dog("초코"), Cat("나비")]         # 종류가 다른 객체
for current_animal in animals:                               # 종류를 묻지 않고 반복
    print(current_animal.name, "→", current_animal.sound())  # 같은 호출

제 이름은 동물입니다. 동물이 소리를 냅니다.
제 이름은 초코입니다. 동물이 소리를 냅니다.
제 이름은 콩이입니다. 멍멍
동물 → 동물이 소리를 냅니다.
초코 → 멍멍
나비 → 야옹


**실행 흐름**

1. 2-1 `super()`로 부모 기능 확장
2. 2-2 `isinstance()` · MRO · `super()` 누락 에러
3. 블록마다 결과 출력

In [10]:
# ---------- 2-1 ----------
# 왜 · 부모 동작을 버리지 않고 super()로 받아 그 위에 덧붙이기 위해
class LoudDog(Dog):                     # Dog를 물려받음
    def sound(self):                    # 부모 sound를 확장
        parent_sound = super().sound()  # 부모(Dog)의 sound() → 멍멍
        return parent_sound + "!!!"     # 덧붙이기


# 왜 · 자식에게 필요한 값만 추가하고 공통 초기화는 부모에게 맡기기 위해
class Bird(Animal):                     # Animal을 물려받음
    def __init__(self, name, can_fly):  # 자식은 인자가 하나 더
        super().__init__(name)          # name 저장은 부모에게 맡김
        self.can_fly = can_fly          # Bird에만 필요한 속성

    def sound(self):   # 오버라이딩
        return "짹짹"  # Bird만의 소리

    # 왜 · 자식에만 있는 속성을 쓰는 기능은 자식 클래스에 두기 위해
    def show_flight(self):                          # 자식에만 있는 메서드
        if self.can_fly:                            # 날 수 있으면
            return f"{self.name}: 날 수 있습니다."  # 원본 문장
        return f"{self.name}: 날 수 없습니다."      # 아니면


# 왜 · super()로 확장한 결과와 추가 속성이 제대로 동작하는지 확인하기 위해
print(LoudDog("두부").sound())                             # 부모 결과 + !!!
bird = Bird("참새", True)                                  # name은 부모, can_fly는 자식이 저장
print(bird.introduce(), bird.sound(), bird.show_flight())  # 물려받은 · 바꾼 · 새 메서드
print(bird.__dict__)                                       # 두 속성 모두 저장됨


# ---------- 2-2 ----------
# 왜 · 자식 객체는 부모 종류로도 인정된다는 것과 메서드 찾는 순서를 확인하기 위해
print(isinstance(dog, Dog), isinstance(dog, Animal), isinstance(cat, Dog))  # 자식 = 부모 객체
print([c.__name__ for c in LoudDog.__mro__])                                # 메서드를 찾는 순서


# 왜 · super().__init__()을 빠뜨리면 부모 속성이 생기지 않는 버그를 재현하기 위해
class BrokenBird(Animal):               # super().__init__()을 빠뜨린 자식
    def __init__(self, name, can_fly):  # 부모 초기화 생략
        self.can_fly = can_fly          # name이 저장되지 않음


try:
    BrokenBird("펭귄", False).introduce()  # name을 쓰는 순간
except AttributeError as e:                # name 속성이 없음
    print("AttributeError:", e)            # 에러 문장

멍멍!!!
제 이름은 참새입니다. 짹짹 참새: 날 수 있습니다.
{'name': '참새', 'can_fly': True}
True True False
['LoudDog', 'Dog', 'Animal', 'object']
AttributeError: 'BrokenBird' object has no attribute 'name'


**실행 흐름**

1. `nn.Module`을 물려받은 `Net` 정의
2. 입력을 넣어 출력 모양 확인
3. `super().__init__()` 누락 모델의 에러 확인

In [11]:
# ---------- 3. 종합 실습 ----------
import torch          # 텐서
from torch import nn  # 층 · 모델


# 1 ~ 3단계: 상속 → super → 층 등록 → forward 오버라이딩
# 왜 · PyTorch 모델이 바로 이 상속 · super · 오버라이딩 규칙으로 만들어진다는 것을 보이기 위해
class Net(nn.Module):              # nn.Module을 물려받음
    def __init__(self):            # 층 준비
        super().__init__()         # 부모(nn.Module) 준비를 먼저
        self.fc = nn.Linear(4, 2)  # 층 등록

    def forward(self, x):  # 오버라이딩: 계산 방법만 정의
        return self.fc(x)  # 입력 4 → 점수 2


# 왜 · 가중치 초깃값을 고정해 같은 PC · 같은 버전에서 출력이 같게 하기 위해
torch.manual_seed(0)                                                       # 가중치 초기값 고정
net = Net()                                                                # 모델 객체
# 상속 확인 · model(x) 출력 모양
print(isinstance(net, nn.Module), tuple(net(torch.ones(3, 4)).shape))
print("물려받은 parameters():", sum(p.numel() for p in net.parameters()))  # 4 × 2 + 2


# 4단계: super().__init__()을 빠뜨린 모델
# 왜 · 모델에서 가장 흔한 실수가 어떤 에러로 나타나는지 미리 보기 위해
class BrokenNet(nn.Module):        # super().__init__() 생략
    def __init__(self):            # 부모 준비 없이
        self.fc = nn.Linear(4, 2)  # 층 저장 → 즉시 에러


try:
    BrokenNet()                  # 만드는 순간
except AttributeError as e:      # 모듈을 등록할 공간이 없음
    print("AttributeError:", e)  # 에러 문장

True (3, 2)
물려받은 parameters(): 10
AttributeError: cannot assign module before Module.__init__() call
